# Stage 3: Hyperparameter tuning (Randomized Search)

Thesis: *Explainable AI for Early Diabetes Prediction* (CDC BRFSS 2015). Config v1.0, master seed 42.

Implements decisions **S9** (Stage 3 plan), **S10** (environment guard, first cell), **S8** (every text-mode `open()` uses UTF-8),
with D5/P4 (search spaces), D14 (KNN and MLP tuned on the 50k subsample, refitted on full development data), S2/S7 (seeds), S1/S5 (frozen index files).

What it does, per model, in the order GNB, DT, LR, SVC, KNN, XGB, LGBM, MLP, CAT, RF:

1. `RandomizedSearchCV` over the 30 frozen configurations x 5 frozen development folds = 150 fits, scored by average precision (AP), `refit=False`, `error_score='raise'`, no time cap, no early stopping of the search.
2. Stops if the evaluated configurations differ from `tc.sampled_configs(model)`.
3. Best configuration = scikit-learn rank 1 of mean CV AP (lowest config_id if tied); refitted on all 202,944 development rows.
4. Saves immediately (checkpoint): cv_results CSV, best-parameters JSON, fitted pipeline (joblib), run record (wall times, machine, versions). A rerun skips completed models.

The test partition is never used: the raw CSV is read only to check its hash and is reduced to the development rows at once.
The AP values printed here are **selection scores only** (KNN and MLP on the subsample); reported model performance comes from the Stage 4 common re-evaluation (D7).

**Run:** put this notebook in `thesis_xai/` next to `thesis_common.py`, `env_setup.py`, `requirements.txt`, `stage1_outputs/` and `stage2_outputs/`, then *Run All*. Expected PC time about 1.6 h, plan for up to 4.5 h. Keep the laptop on mains power and stop it from sleeping. If it stops, just run all again: finished models are skipped.

In [1]:
# Cell 1 - environment guard (S10): runs before numpy / scikit-learn are imported.
import os
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:                                   # fallback machine only (S9)
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/thesis_xai"   # edit if your Drive folder differs
else:
    PROJECT_DIR = os.path.abspath(".")         # the PC: the folder that holds this notebook
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import env_setup
env_setup.ensure_pinned(PROJECT_DIR, install=IN_COLAB)

All pinned packages present at the pinned versions.


{}

In [2]:
# Cell 2 - configuration
import hashlib
import json
import platform
import time
import warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import RandomizedSearchCV

import thesis_common as tc

MACHINE = "colab" if IN_COLAB else "pc"
STAGE = "stage3_tuning.ipynb"
RUN_ORDER = ["GNB", "DT", "LR", "SVC", "KNN", "XGB", "LGBM", "MLP", "CAT", "RF"]   # S9, cheapest first
MODELS_TO_RUN = RUN_ORDER          # all ten (S9); do not change for the thesis run
SEARCH_N_JOBS_PARALLEL = 6         # S9: LR, DT, SVC, GNB search 6 folds in parallel (6 physical cores)
SCORING = "average_precision"      # Ch3 3.4.2; LinearSVC is scored by its decision function

# Expected Stage 1 quantities (registry S1-025, S1-026, S1-031, S1-033, S1-034, S1-060)
EXPECTED = {"dev_rows": 202944, "dev_pos": 28277, "dev_fold_sizes": [40589, 40589, 40589, 40589, 40588],
            "sub_rows": 50000, "sub_pos": 6965, "sub_fold_sizes": [10000] * 5, "n_raw": 253680}

OUT = os.path.join(PROJECT_DIR, "stage3_outputs")
MODEL_DIR = os.path.join(OUT, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

assert sorted(RUN_ORDER) == sorted(tc.MODELS), "RUN_ORDER must contain each of the ten models once"
assert tc.SEEDS["search_sampling"] == 1333843299 and tc.MODEL_SEED == 22980462, "seed derivation changed (S2/S7)"
assert tc.N_ITER == 30 and tc.N_FOLDS == 5

print(f"machine={MACHINE}  python={platform.python_version()}  scikit-learn={sklearn.__version__}  "
      f"cpu_count={os.cpu_count()}")
print(f"search_sampling seed={tc.SEEDS['search_sampling']}  estimator seed={tc.MODEL_SEED}")
print("output folder:", OUT)

machine=pc  python=3.13.0  scikit-learn=1.6.1  cpu_count=12
search_sampling seed=1333843299  estimator seed=22980462
output folder: C:\Users\USER\Downloads\thesis_xai\stage3_outputs


In [3]:
# Cell 3 - the frozen search configurations must be byte-identical before any search (S7, S9).
# Nothing is written to the search_configs folder; a missing or different file stops the run.
def find_config_dir():
    for d in [os.path.join(PROJECT_DIR, "stage2_outputs", "search_configs"),
              os.path.join(PROJECT_DIR, "search_configs")]:
        if os.path.isdir(d):
            return d
    raise SystemExit("search_configs folder not found in stage2_outputs/ or the project folder. "
                     "Copy the Stage 2 search_configs folder there.")


def config_file(cfg_dir, model):
    names = [f for f in os.listdir(cfg_dir) if f.lower().endswith(".csv")]
    stem = lambda f: os.path.splitext(f)[0].upper()
    hits = [f for f in names if stem(f) == model or stem(f).endswith("_" + model) or stem(f).startswith(model + "_")]
    if len(hits) != 1:
        raise SystemExit(f"Expected exactly one search_configs file for {model}, found {hits} in {cfg_dir}")
    return os.path.join(cfg_dir, hits[0])


CFG_DIR = find_config_dir()
rows = []
for m in RUN_ORDER:
    path = config_file(CFG_DIR, m)
    expected = tc.configs_frame(m).to_csv(index=False, lineterminator="\n", float_format="%.10g").encode()
    with open(path, "rb") as f:
        actual = f.read()
    rows.append({"model": m, "file": os.path.basename(path), "n_configs": len(tc.sampled_configs(m)),
                 "sha256": hashlib.sha256(actual).hexdigest(),
                 "status": "verified: identical" if actual == expected else "MISMATCH"})
cfg_check = pd.DataFrame(rows)
cfg_check.to_csv(os.path.join(OUT, f"search_configs_check_{MACHINE}.csv"), index=False, encoding="utf-8")
print(cfg_check.to_string(index=False))
if (cfg_check["status"] != "verified: identical").any():
    raise SystemExit("A search configuration file differs from tc.sampled_configs(). Stop: restore the frozen files.")
print(f"\n{len(cfg_check)}/10 configuration files verified byte-identical.")

model                    file  n_configs                                                           sha256              status
  GNB  search_configs_GNB.csv         30 937143e666e05b4e3c126ac189f141a9fc0fed44de024515ae8c45723e6670fc verified: identical
   DT   search_configs_DT.csv         30 5509962d584532ce3e94de9bc9997aa7d1d341d5c1ac2246e44a69551c001aed verified: identical
   LR   search_configs_LR.csv         30 6e57d88526c13b6d641b5ebc8f47c53b14108da1171283ecdcdf0352ff470814 verified: identical
  SVC  search_configs_SVC.csv         30 95b25eb50f13c8b9324dd76c61556d7897a50a97e78793584ccac90b17067135 verified: identical
  KNN  search_configs_KNN.csv         30 52c566a8247103a66c437869aebe389c135859c1ad08d942d829dd78589273cf verified: identical
  XGB  search_configs_XGB.csv         30 9bae2e4ca16ccfba4c721bd460c2bd103879cb018860348bc0767613fd1c7d27 verified: identical
 LGBM search_configs_LGBM.csv         30 72f5b51f5d3087fff2a1cafc1c7fa0e9d544275142a0a957a4da2e2fa2492b5a verified: id

In [4]:
# Cell 4 - data and frozen folds (S1a/S5: index files are hash-checked, never recomputed).
ROW_ID_NAMES = ["row_id", "row", "idx", "index"]


def row_ids(df, name):
    col = next((c for c in ROW_ID_NAMES if c in df.columns), None)
    if col is None or "fold" not in df.columns:
        raise SystemExit(f"{name}: expected a row-id column ({ROW_ID_NAMES}) and 'fold'; found {list(df.columns)}")
    r = df[col].to_numpy()
    if not (np.issubdtype(r.dtype, np.integer) and r.min() >= 0 and r.max() < EXPECTED["n_raw"]
            and len(np.unique(r)) == len(r)):
        raise SystemExit(f"{name}: column '{col}' is not a set of unique raw-row positions")
    return r


X_all, y_all = tc.load_data(PROJECT_DIR)                 # SHA-256 checked against the canonical file
assert len(X_all) == EXPECTED["n_raw"]
idx = tc.load_indices(PROJECT_DIR)                        # every file checked against MANIFEST.csv
dev_folds = idx["dev_folds.csv"]
sub_folds = idx["dev_subsample_50k.csv"]
del idx                                                   # nothing else from Stage 1 is needed here

dev_rows = row_ids(dev_folds, "dev_folds.csv")
sub_rows = row_ids(sub_folds, "dev_subsample_50k.csv")

# Development data only; the test rows are dropped here and never used in Stage 3.
X_dev = X_all.iloc[dev_rows].reset_index(drop=True)
y_dev = y_all[dev_rows]
pos_in_dev = pd.Series(np.arange(len(dev_rows)), index=dev_rows)
sub_pos = pos_in_dev.loc[sub_rows].to_numpy()             # KeyError if a subsample row is not a development row
X_sub = X_dev.iloc[sub_pos].reset_index(drop=True)
y_sub = y_dev[sub_pos]
del X_all, y_all

for df, rr, name in [(dev_folds, dev_rows, "dev_folds.csv"), (sub_folds, sub_rows, "dev_subsample_50k.csv")]:
    tcol = next((c for c in [tc.TARGET, "target", "y", "label"] if c in df.columns), None)
    if tcol is not None:   # labels stored with the index must match the raw file
        yy = y_dev if name == "dev_folds.csv" else y_sub
        assert np.array_equal(df[tcol].to_numpy().astype(int), yy), f"{name}: stored labels differ from the data"

cv_dev = tc.fold_splits(dev_folds)
cv_sub = tc.fold_splits(sub_folds)

checks = {
    "development rows": (len(X_dev), EXPECTED["dev_rows"]),
    "development positives": (int(y_dev.sum()), EXPECTED["dev_pos"]),
    "development fold sizes": ([len(v) for _, v in cv_dev], EXPECTED["dev_fold_sizes"]),
    "subsample rows": (len(X_sub), EXPECTED["sub_rows"]),
    "subsample positives": (int(y_sub.sum()), EXPECTED["sub_pos"]),
    "subsample fold sizes": ([len(v) for _, v in cv_sub], EXPECTED["sub_fold_sizes"]),
    "subsample keeps development fold (S4)": (
        bool(np.array_equal(sub_folds["fold"].to_numpy(), dev_folds["fold"].to_numpy()[sub_pos])), True),
}
for k, (got, want) in checks.items():
    print(f"{'PASS' if got == want else 'FAIL'}  {k}: {got}")
if any(got != want for got, want in checks.values()):
    raise SystemExit("Stage 1 data checks failed; do not tune.")

PASS  development rows: 202944
PASS  development positives: 28277
PASS  development fold sizes: [40589, 40589, 40589, 40589, 40588]
PASS  subsample rows: 50000
PASS  subsample positives: 6965
PASS  subsample fold sizes: [10000, 10000, 10000, 10000, 10000]
PASS  subsample keeps development fold (S4): True


In [5]:
# Cell 5 - one model: search, check, select, refit, save (checkpoint).
def py(v):
    # numpy scalar -> Python scalar, so repr() is a plain literal (ast.literal_eval restores it exactly)
    return v.item() if hasattr(v, "item") else v


def write_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)


def done_path(m):
    return os.path.join(MODEL_DIR, m, f"run_record_{m}.json")


def run_model(m):
    d = os.path.join(MODEL_DIR, m)
    os.makedirs(d, exist_ok=True)
    if os.path.exists(done_path(m)):
        with open(done_path(m), encoding="utf-8") as f:
            rec = json.load(f)
        print(f"{m}: already completed on {rec['machine']} at {rec['finished_utc']} - skipped")
        return rec

    sub = m in tc.SUBSAMPLE_TUNED
    X, y, cv = (X_sub, y_sub, cv_sub) if sub else (X_dev, y_dev, cv_dev)
    multi = m in tc.MULTITHREADED
    search_n_jobs = 1 if multi else SEARCH_N_JOBS_PARALLEL       # no nested parallelism (S9)
    expected = tc.sampled_configs(m)

    search = RandomizedSearchCV(tc.make_pipeline(m, n_jobs=-1), tc.SEARCH_SPACES[m], n_iter=tc.N_ITER,
                                scoring=SCORING, cv=cv, refit=False, error_score="raise",
                                random_state=tc.SEEDS["search_sampling"], n_jobs=search_n_jobs, verbose=1)
    started = datetime.now(timezone.utc)
    print(f"\n=== {m} ({tc.MODEL_NAMES[m]}) | {'50k subsample' if sub else 'full development'} "
          f"| {len(expected)} configs x {len(cv)} folds | search n_jobs={search_n_jobs} | start {started:%H:%M:%S} UTC")
    with warnings.catch_warnings(record=True) as w_search:
        warnings.simplefilter("always")
        t0 = time.perf_counter()
        search.fit(X, y)
        search_s = time.perf_counter() - t0
    res = search.cv_results_

    # The evaluated configurations must be exactly the frozen ones, in order (S9).
    if len(res["params"]) != len(expected) or any(a != b for a, b in zip(res["params"], expected)):
        raise SystemExit(f"{m}: evaluated configurations differ from tc.sampled_configs(); results not saved.")

    cvr = tc.configs_frame(m)                                   # config_id + parameters, as frozen
    split_cols = [f"split{k}_test_score" for k in range(len(cv))]
    for c in split_cols + ["mean_test_score", "std_test_score", "rank_test_score",
                           "mean_fit_time", "std_fit_time", "mean_score_time", "std_score_time"]:
        cvr[c] = res[c]
    cvr = cvr.rename(columns=lambda c: c.replace("test_score", "test_ap"))
    if not np.isfinite(cvr[[c.replace("test_score", "test_ap") for c in split_cols]].to_numpy()).all():
        raise SystemExit(f"{m}: non-finite fold scores")
    cvr.to_csv(os.path.join(d, f"cv_results_{m}.csv"), index=False, encoding="utf-8", float_format="%.10g")

    rank1 = np.flatnonzero(res["rank_test_score"] == 1)
    best = int(rank1[0])                                        # = scikit-learn best_index_ (lowest id on ties)
    best_params = {k: py(v) for k, v in res["params"][best].items()}
    best_rec = {"model": m, "model_name": tc.MODEL_NAMES[m], "config_id": best, "n_rank1_ties": int(len(rank1)),
                "params": {k: (list(v) if isinstance(v, tuple) else v) for k, v in best_params.items()},
                "params_repr": {k: repr(v) for k, v in best_params.items()},
                "mean_cv_ap": float(res["mean_test_score"][best]), "sd_cv_ap": float(res["std_test_score"][best]),
                "fold_ap": [float(res[c][best]) for c in split_cols],
                "tuned_on": "dev_subsample_50k.csv (D14)" if sub else "dev_folds.csv",
                "tuning_rows": int(len(X)), "n_configs": len(expected), "n_folds": len(cv),
                "scoring": SCORING, "note": "selection score only; reported performance comes from Stage 4 (D7)"}
    write_json(best_rec, os.path.join(d, f"best_params_{m}.json"))

    # Refit the best configuration on the full development set (202,944 rows, incl. KNN and MLP, D14).
    pipe = tc.make_pipeline(m, n_jobs=-1).set_params(**best_params)
    with warnings.catch_warnings(record=True) as w_refit:
        warnings.simplefilter("always")
        t0 = time.perf_counter()
        pipe.fit(X_dev, y_dev)
        refit_s = time.perf_counter() - t0
    pkl = os.path.join(d, f"pipeline_{m}.joblib")
    joblib.dump(pipe, pkl + ".tmp", compress=3)
    os.replace(pkl + ".tmp", pkl)

    def warn_list(ws):
        return sorted({f"{x.category.__name__}: {str(x.message)[:200]}" for x in ws})

    rec = {"model": m, "stage": STAGE, "config_version": tc.CONFIG_VERSION, "machine": MACHINE,
           "search_time_reportable": MACHINE == "pc",            # Colab timings are not reported (D14, S9)
           "started_utc": started.isoformat(timespec="seconds"),
           "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "search_wall_s": round(search_s, 2), "refit_wall_s": round(refit_s, 2),
           "fits": len(expected) * len(cv), "search_n_jobs": search_n_jobs,
           "estimator_n_jobs": -1 if multi else None, "refit_rows": int(len(X_dev)),
           "best_config_id": best, "best_mean_cv_ap": best_rec["mean_cv_ap"], "best_sd_cv_ap": best_rec["sd_cv_ap"],
           "warnings_search_main_process": warn_list(w_search),
           "warnings_search_note": ("captured" if search_n_jobs == 1 else
                                    "folds ran in worker processes; their warnings are not captured here"),
           "warnings_refit": warn_list(w_refit),
           "pipeline_file": os.path.basename(pkl), "pipeline_mb": round(os.path.getsize(pkl) / 2**20, 1),
           "pipeline_sha256": tc.sha256_file(pkl), "environment": tc.environment(MACHINE)}
    write_json(rec, done_path(m))                               # written last = the checkpoint marker
    print(f"{m}: search {search_s / 60:.1f} min, refit {refit_s:.1f} s | best config {best} "
          f"(mean CV AP {rec['best_mean_cv_ap']:.4f} +/- {rec['best_sd_cv_ap']:.4f}, rank-1 ties {len(rank1)}) | "
          f"warnings: search {len(rec['warnings_search_main_process'])}, refit {len(rec['warnings_refit'])}")
    return rec

In [6]:
# Cell 6 - run all models (long). Safe to interrupt and rerun: finished models are skipped.
t_all = time.perf_counter()
records = {m: run_model(m) for m in MODELS_TO_RUN}
print(f"\nThis session: {(time.perf_counter() - t_all) / 3600:.2f} h")


=== GNB (Gaussian Naive Bayes) | full development | 30 configs x 5 folds | search n_jobs=6 | start 15:46:22 UTC
Fitting 5 folds for each of 30 candidates, totalling 150 fits
GNB: search 0.2 min, refit 0.1 s | best config 16 (mean CV AP 0.3505 +/- 0.0042, rank-1 ties 1) | warnings: search 0, refit 0

=== DT (Decision Tree) | full development | 30 configs x 5 folds | search n_jobs=6 | start 15:46:32 UTC
Fitting 5 folds for each of 30 candidates, totalling 150 fits
DT: search 0.4 min, refit 0.8 s | best config 3 (mean CV AP 0.4066 +/- 0.0081, rank-1 ties 1) | warnings: search 0, refit 0

=== LR (Logistic Regression) | full development | 30 configs x 5 folds | search n_jobs=6 | start 15:46:55 UTC
Fitting 5 folds for each of 30 candidates, totalling 150 fits
LR: search 3.8 min, refit 1.3 s | best config 20 (mean CV AP 0.4058 +/- 0.0042, rank-1 ties 1) | warnings: search 0, refit 0

=== SVC (Linear SVC) | full development | 30 configs x 5 folds | search n_jobs=6 | start 15:50:44 UTC
Fitting

In [7]:
# Cell 7 - summary, report and environment log
summ = []
for m in RUN_ORDER:
    if not os.path.exists(done_path(m)):
        summ.append({"model": m, "status": "not run"})
        continue
    with open(done_path(m), encoding="utf-8") as f:
        r = json.load(f)
    with open(os.path.join(MODEL_DIR, m, f"best_params_{m}.json"), encoding="utf-8") as f:
        b = json.load(f)
    summ.append({"model": m, "status": "done", "machine": r["machine"], "tuned_on": b["tuned_on"],
                 "tuning_rows": b["tuning_rows"], "best_config_id": b["config_id"],
                 "n_rank1_ties": b["n_rank1_ties"], "mean_cv_ap": round(b["mean_cv_ap"], 4),
                 "sd_cv_ap": round(b["sd_cv_ap"], 4),
                 "search_min": round(r["search_wall_s"] / 60, 2) if r["search_time_reportable"] else None,
                 "refit_s": round(r["refit_wall_s"], 1) if r["search_time_reportable"] else None,
                 "warnings_search": len(r["warnings_search_main_process"]), "warnings_refit": len(r["warnings_refit"]),
                 "pipeline_mb": r["pipeline_mb"], "pipeline_sha256": r["pipeline_sha256"]})
summary = pd.DataFrame(summ)
summary.to_csv(os.path.join(OUT, "stage3_summary.csv"), index=False, encoding="utf-8")
print(summary.drop(columns=["pipeline_sha256"], errors="ignore").to_string(index=False))

done = summary[summary["status"] == "done"]
total_h = sum(json.load(open(done_path(m), encoding="utf-8"))["search_wall_s"] for m in done["model"]) / 3600
env = tc.environment(MACHINE)
write_json({**env, "stage": STAGE, "written_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "pins_differ": {k: list(v) for k, v in tc.check_pins(PROJECT_DIR).items()}},
           os.path.join(OUT, f"env_log_stage3_{MACHINE}.json"))

lines = ["# Stage 3 tuning report", "",
         f"Generated by `{STAGE}` (config {tc.CONFIG_VERSION}) on {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC. "
         f"Machine(s): {', '.join(sorted(done['machine'].unique())) if len(done) else '-'}. "
         f"Python {env['python']}, scikit-learn {env['packages']['scikit-learn']}.", "",
         f"Models completed: {len(done)}/10. Total search wall time (all machines): {total_h:.2f} h. "
         "Mean CV AP is the selection score only (KNN and MLP: 50k subsample); "
         "reported performance comes from the Stage 4 re-evaluation (D7).", "",
         "| Model | Machine | Tuned on | Best config | Mean CV AP (SD) | Search min | Refit s | Warnings (search/refit) |",
         "| --- | --- | --- | --- | --- | --- | --- | --- |"]
for r in summ:
    if r["status"] != "done":
        lines.append(f"| {r['model']} | not run | | | | | | |")
        continue
    lines.append(f"| {r['model']} | {r['machine']} | {r['tuning_rows']:,} rows | {r['best_config_id']} "
                 f"| {r['mean_cv_ap']:.4f} ({r['sd_cv_ap']:.4f}) | {r['search_min'] if r['search_min'] is not None else 'n/r'} "
                 f"| {r['refit_s'] if r['refit_s'] is not None else 'n/r'} | {r['warnings_search']}/{r['warnings_refit']} |")
lines += ["", "n/r = run on Colab, time not reported (D14). Worker-process warnings for LR, DT, SVC and GNB "
          "searches are not captured (folds ran in parallel processes)."]
with open(os.path.join(OUT, "stage3_report.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print(f"\nCompleted {len(done)}/10. Files to send back: stage3_outputs/stage3_summary.csv, stage3_report.md, "
      f"env_log_stage3_{MACHINE}.json, search_configs_check_{MACHINE}.csv, models/*/best_params_*.json, "
      "models/*/run_record_*.json, models/*/cv_results_*.csv (not the .joblib files).")

model status machine                    tuned_on  tuning_rows  best_config_id  n_rank1_ties  mean_cv_ap  sd_cv_ap  search_min  refit_s  warnings_search  warnings_refit  pipeline_mb
  GNB   done      pc               dev_folds.csv       202944              16             1      0.3505    0.0042        0.15      0.1                0               0          0.0
   DT   done      pc               dev_folds.csv       202944               3             1      0.4066    0.0081        0.37      0.8                0               0          0.1
   LR   done      pc               dev_folds.csv       202944              20             1      0.4058    0.0042        3.79      1.3                0               0          0.0
  SVC   done      pc               dev_folds.csv       202944              20             1      0.4055    0.0041        3.71      1.2                0               0          0.0
  KNN   done      pc dev_subsample_50k.csv (D14)        50000              16             1    